In [1]:
# %pip install pandas numpy

In [2]:
import pandas as pd
import numpy as np

file_id = '12FsR70V30i08i5uAsGQ1zxR-jS7sInEh'   # new file
url = f"https://drive.usercontent.google.com/download?id={file_id}&export=download&confirm=t"

df_policy = pd.read_csv(url)
df_policy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 391166 entries, 0 to 391165
Data columns (total 72 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   loan_amnt                    391166 non-null  int64  
 1   term                         391166 non-null  int64  
 2   int_rate                     391166 non-null  float64
 3   installment                  391166 non-null  float64
 4   sub_grade                    391166 non-null  object 
 5   emp_length                   391166 non-null  float64
 6   home_ownership               391166 non-null  object 
 7   annual_inc                   391166 non-null  float64
 8   verification_status          391166 non-null  object 
 9   purpose                      391166 non-null  object 
 10  dti                          391166 non-null  float64
 11  delinq_2yrs                  391166 non-null  float64
 12  inq_last_6mths               391166 non-null  float64
 13 

### Create Simulated Loan Size Grid

In [3]:
# Loan size multipliers
loan_size_multipliers = [0.50, 0.75, 1.00, 1.25]

# Add borrower ID if needed
df_policy = df_policy.reset_index(drop=False).rename(columns={"index": "borrower_id"})

grid_list = []

for mult in loan_size_multipliers:
    temp = df_policy.copy()
    temp["loan_size_multiplier"] = mult
    temp["sim_loan_amnt"] = (temp["loan_amnt"] * mult).round(2)
    
    # Recalculate key ratios
    temp["sim_income_to_loan_ratio"] = temp["annual_inc"] / temp["sim_loan_amnt"]
    temp["sim_loan_to_income_ratio"] = temp["sim_loan_amnt"] / temp["annual_inc"]
    
    grid_list.append(temp)

loan_grid_df = pd.concat(grid_list, ignore_index=True)

# Clean up infinities
loan_grid_df.replace([np.inf, -np.inf], np.nan, inplace=True)

loan_grid_df.head()

,level_0,loan_amnt,term,int_rate,installment,sub_grade,emp_length,home_ownership,annual_inc,verification_status,...,total_interest,yield_on_principal,expected_lgd,expected_revenue,expected_loss,expected_profit,loan_size_multiplier,sim_loan_amnt,sim_income_to_loan_ratio,sim_loan_to_income_ratio
0,0,2550,36,16.55,90.35,D2,3.0,RENT,72500.0,Source Verified,...,702.40,0.2755,0.655482,599.996798,243.685645,356.311153,0.5,1275.0,56.862745,0.017586
1,1,9600,36,13.99,328.06,C3,1.0,MORTGAGE,73000.0,Source Verified,...,2210.11,0.2302,0.621307,1838.385358,1003.193284,835.192074,0.5,4800.0,15.208333,0.065753
2,2,21000,36,12.35,701.02,B4,2.0,MORTGAGE,78000.0,Source Verified,...,4236.58,0.2017,0.574821,3796.638399,1253.521381,2543.117018,0.5,10500.0,7.428571,0.134615
3,3,6000,36,17.86,216.50,D5,10.0,RENT,50000.0,Verified,...,1793.76,0.2990,0.683216,1302.134949,1123.515798,178.619152,0.5,3000.0,16.666667,0.060000
4,4,20000,60,19.19,520.91,E3,1.0,RENT,40000.0,Verified,...,11254.25,0.5627,0.701859,5477.192799,7205.598633,-1728.405834,0.5,10000.0,4.000000,0.250000


### Recalculate expected profit for each simulated loan size

In [4]:
# calculate loan revenue

def calculate_loan_revenue(loan_amnt, int_rate, term):
    """
    Calculate loan revenue metrics.
    
    Parameters
    ----------
    loan_amnt : float — principal in dollars
    int_rate  : float — annual interest rate as percentage (e.g. 13.5)
    term      : int   — term in months (36 or 60)
    
    Returns
    -------
    dict: 
    total_interest -> gross interest revenue earned over the loan lifetime (total repaid minus principal).
    yield_on_principal -> total interest as a fraction of loan amount, enables profitability comparison across loan sizes.
    
    """
    r = int_rate / 100 / 12

    if r == 0:
        installment = loan_amnt / term
    else:
        installment = loan_amnt * r * (1 + r)**term / ((1 + r)**term - 1)

    total_repaid = installment * term
    total_interest = total_repaid - loan_amnt

    return {
        "total_interest":     round(total_interest, 2),
        "yield_on_principal": round(total_interest / loan_amnt, 4),
    }

# Apply to dataframe
sim_revenue_cols = loan_grid_df.apply(
    lambda row: calculate_loan_revenue(row["sim_loan_amnt"], row["int_rate"], row["term"]),
    axis=1,
    result_type="expand"
)

# Rename columns
sim_revenue_cols = sim_revenue_cols.rename(columns={
    "total_interest": "sim_total_interest",
    "yield_on_principal": "sim_yield_on_principal"
})

# merge with loan_frid_df
loan_grid_df = pd.concat([loan_grid_df, sim_revenue_cols], axis=1)

In [5]:
loan_grid_df[['sim_total_interest', 'sim_yield_on_principal']].head()

,sim_total_interest,sim_yield_on_principal
0,351.20,0.2755
1,1105.06,0.2302
2,2118.29,0.2017
3,896.88,0.2990
4,5627.13,0.5627


In [6]:
# Simulated expected revenue if borrower does not default
loan_grid_df["sim_expected_revenue"] = (1 - loan_grid_df["predicted_prob"]) * loan_grid_df["sim_total_interest"]

# Simulated expected loss if borrower defaults
loan_grid_df["sim_expected_loss"] = loan_grid_df["predicted_prob"] * loan_grid_df["expected_lgd"] * loan_grid_df["sim_loan_amnt"]

# simulated expected profit = expected revenue - expected loss
loan_grid_df["sim_expected_profit"] = loan_grid_df["sim_expected_revenue"] - loan_grid_df["sim_expected_loss"]

loan_grid_df[
    [
        "predicted_prob",
        "sim_total_interest",
        "expected_lgd",
        "sim_expected_revenue",
        "sim_expected_loss",
        "sim_expected_profit"
    ]
].head()

,predicted_prob,sim_total_interest,expected_lgd,sim_expected_revenue,sim_expected_loss,sim_expected_profit
0,0.145790,351.20,0.655482,299.998399,121.842823,178.155577
1,0.168193,1105.06,0.621307,919.196838,501.596642,417.600196
2,0.103844,2118.29,0.574821,1898.319199,626.760690,1271.558509
3,0.274075,896.88,0.683216,651.067475,561.757899,89.309576
4,0.513322,5627.13,0.701859,2738.598833,3602.799316,-864.200483


### Introduce Risk-Adjusted Objective Function

In [ ]:
ALPHA = 0.5 #moderate penalty

# Risk-adjusted profit: discounts expected profit by default probability
# Intuition: same profit from a risky borrower is worth less than from a safe one

loan_grid_df["risk_adjusted_profit"] = (
    loan_grid_df["sim_expected_profit"] / (1 + ALPHA * loan_grid_df["predicted_prob"])
)

# Sharpe-style score: profit per unit of risk (PD as proxy for volatility)
# Rewards high profit AND low risk simultaneously

loan_grid_df["profit_risk_ratio"] = (
    loan_grid_df["sim_expected_profit"] / (loan_grid_df["predicted_prob"] + 1e-6)
)

# Composite objective: combines risk-adjusted profit + affordability constraint
# installment_to_income_ratio penalizes loans that overburden the borrower

loan_grid_df["objective_score"] = (
    loan_grid_df["risk_adjusted_profit"]
    * (1 - loan_grid_df["installment_to_income_ratio"].clip(upper=1))
)

# check
loan_grid_df[[
    "sim_loan_amnt", "predicted_prob", "sim_expected_profit",
    "risk_adjusted_profit", "profit_risk_ratio", "objective_score"
]].head()

,sim_loan_amnt,predicted_prob,sim_expected_profit,risk_adjusted_profit,profit_risk_ratio,objective_score
0,1275.0,0.145790,178.155577,166.051236,1221.989317,163.568025
1,4800.0,0.168193,417.600196,385.205773,2482.850957,364.432523
2,10500.0,0.103844,1271.558509,1208.795678,12244.823985,1078.427994
3,3000.0,0.274075,89.309576,78.545844,325.856788,74.464602
4,10000.0,0.513322,-864.200483,-687.695718,-1683.540426,-580.227445


In [ ]:
# Select loan size that maximizes objective score per borrower
optimal_loans = loan_grid_df.loc[
    loan_grid_df.groupby("borrower_id")["objective_score"].idxmax()
].copy()

# Apply business constraints
# Keep only profitable loans
optimal_loans = optimal_loans[
    optimal_loans["sim_expected_profit"] > 0
]

# Risk threshold (adjust if needed)
optimal_loans = optimal_loans[
    optimal_loans["predicted_prob"] < 0.4
]

# output dataset
final_policy = optimal_loans[[
    "borrower_id",
    "predicted_prob",
    "loan_amnt",
    "sim_loan_amnt",
    "loan_size_multiplier",
    "sim_expected_profit",
    "objective_score"
]]

final_policy.head()